# Stage 2 CNN with U-Net Architecture

This notebook evaluates a 1D U-Net architecture for P-wave arrival-time picking using the existing processed Stage 2 dataset. The training, validation, and test splits are unchanged from the baseline CNN experiment.

## Imports and dataset paths

The existing processed Stage 2 waveform datasets are reused without modification.

In [15]:
import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

processed_dir = Path("../data/processed/stage2_cnn")

train_processed_path = processed_dir / "train_waveforms.h5"
val_processed_path = processed_dir / "validation_waveforms.h5"
test_processed_path = processed_dir / "test_waveforms.h5"

print("Device:", device)
print("Processed data directory:", processed_dir)

Device: cpu
Processed data directory: ../data/processed/stage2_cnn


## Load metadata and recreate dataset splits

The earthquake metadata are loaded and divided by `source_id` using the same train, validation, and test split as the baseline Stage 2 CNN.

In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split

metadata_path = (
    "../data/sample/metadata/"
    "metadata_Instance_events_10k.csv"
)

metadata = pd.read_csv(metadata_path)

print("Full metadata shape:", metadata.shape)

Full metadata shape: (10000, 115)


### Create source-based train, validation, and test splits

The split is performed using `source_id` to prevent recordings from the same earthquake source from appearing across multiple subsets.

In [17]:
from sklearn.model_selection import GroupShuffleSplit

metadata = metadata.reset_index(drop=True)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(
    gss.split(
        metadata,
        groups=metadata["source_id"]
    )
)

train_metadata = metadata.iloc[train_idx].reset_index(drop=True)
temp_metadata = metadata.iloc[temp_idx].reset_index(drop=True)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(
    gss.split(
        temp_metadata,
        groups=temp_metadata["source_id"]
    )
)

val_metadata = temp_metadata.iloc[val_idx].reset_index(drop=True)
test_metadata = temp_metadata.iloc[test_idx].reset_index(drop=True)

print("Train:", len(train_metadata))
print("Validation:", len(val_metadata))
print("Test:", len(test_metadata))

Train: 6761
Validation: 1593
Test: 1646


## Create time-localized P-wave targets

A Gaussian target is generated around the annotated P-wave arrival. At the 100 Hz sampling rate, a standard deviation of 30 samples corresponds to 0.30 seconds.

In [18]:
import numpy as np


def create_p_wave_target(
    p_sample,
    n_samples=12000,
    sigma_samples=30
):
    samples = np.arange(n_samples)

    target = np.exp(
        -0.5 * (
            (samples - p_sample) / sigma_samples
        ) ** 2
    )

    return target.astype(np.float32)

## Load preprocessed waveform datasets

The existing processed waveforms are reused. P-wave targets use a Gaussian with a standard deviation of 30 samples, corresponding to 0.30 seconds at the 100 Hz sampling rate.

In [19]:
import h5py
import numpy as np
import torch
from torch.utils.data import Dataset


class PWaveDataset(Dataset):

    def __init__(
        self,
        waveform_h5_path,
        split_metadata,
        n_samples=12000,
        sigma_samples=30
    ):
        self.waveform_h5_path = waveform_h5_path
        self.metadata = split_metadata.reset_index(drop=True)
        self.n_samples = n_samples
        self.sigma_samples = sigma_samples

        self.h5_file = None

    def _open_file(self):

        if self.h5_file is None:
            self.h5_file = h5py.File(
                self.waveform_h5_path,
                "r"
            )

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):

        self._open_file()

        row = self.metadata.iloc[idx]

        waveform = np.asarray(
            self.h5_file["waveforms"][idx],
            dtype=np.float32
        )

        p_sample = int(
            row["trace_P_arrival_sample"]
        )

        target = create_p_wave_target(
            p_sample=p_sample,
            n_samples=self.n_samples,
            sigma_samples=self.sigma_samples
        )

        return (
            torch.tensor(
                waveform,
                dtype=torch.float32
            ),
            torch.tensor(
                target,
                dtype=torch.float32
            )
        )

## Create train, validation, and test datasets

The existing processed waveform files are paired with the corresponding metadata splits. P-wave targets use a Gaussian standard deviation of 30 samples, corresponding to 0.30 seconds at 100 Hz.

In [20]:
train_dataset = PWaveDataset(
    waveform_h5_path=train_processed_path,
    split_metadata=train_metadata,
    sigma_samples=30
)

val_dataset = PWaveDataset(
    waveform_h5_path=val_processed_path,
    split_metadata=val_metadata,
    sigma_samples=30
)

test_dataset = PWaveDataset(
    waveform_h5_path=test_processed_path,
    split_metadata=test_metadata,
    sigma_samples=30
)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

Train dataset: 6761
Validation dataset: 1593
Test dataset: 1646


## Create DataLoaders

Training batches are shuffled, while validation and test batches preserve metadata order for evaluation.

In [21]:
from torch.utils.data import DataLoader

batch_size = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 846
Validation batches: 200
Test batches: 206


## Calculate positive-class weight

The positive-class weight compensates for the strong imbalance between P-wave target samples and non-P-wave samples.

In [22]:
target_fraction = 0.0041773343

pos_weight_value = 1.0 / target_fraction

pos_weight = torch.tensor(
    [pos_weight_value],
    dtype=torch.float32,
    device=device
)

print("Target positive fraction:", target_fraction)
print("Positive weight:", pos_weight_value)

Target positive fraction: 0.0041773343
Positive weight: 239.38711345175318


## Define the 1D U-Net

The 1D U-Net uses encoder-decoder layers with skip connections to preserve high-resolution temporal information during P-wave localization.

In [29]:
class UNet1D(nn.Module):

    def __init__(self):
        super().__init__()

        # Encoder
        self.enc1 = nn.Sequential(
            nn.Conv1d(3, 16, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(4)
        )

        self.enc2 = nn.Sequential(
            nn.Conv1d(16, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(4)
        )

        self.enc3 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv1d(64, 64, kernel_size=5, padding=2),
            nn.ReLU()
        )

        # Decoder
        self.up3 = nn.ConvTranspose1d(
            64, 64, kernel_size=2, stride=2
        )

        # 64 + 32 = 96
        self.dec3 = nn.Sequential(
            nn.Conv1d(96, 64, kernel_size=5, padding=2),
            nn.ReLU()
        )

        self.up2 = nn.ConvTranspose1d(
            64, 32, kernel_size=4, stride=4
        )

        # 32 + 16 = 48
        self.dec2 = nn.Sequential(
            nn.Conv1d(48, 32, kernel_size=5, padding=2),
            nn.ReLU()
        )

        self.up1 = nn.ConvTranspose1d(
            32, 16, kernel_size=4, stride=4
        )

        # 16 + 3 = 19
        self.dec1 = nn.Sequential(
            nn.Conv1d(19, 16, kernel_size=5, padding=2),
            nn.ReLU()
        )

        self.output = nn.Conv1d(
            16, 1, kernel_size=1
        )

    def forward(self, x):

        e1 = self.enc1(x)      # [B, 16, 3000]
        e2 = self.enc2(e1)     # [B, 32, 750]
        e3 = self.enc3(e2)     # [B, 64, 375]

        b = self.bottleneck(e3)

        d3 = self.up3(b)       # [B, 64, 750]
        d3 = torch.cat([d3, e2], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)      # [B, 32, 3000]
        d2 = torch.cat([d2, e1], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)      # [B, 16, 12000]
        d1 = torch.cat([d1, x], dim=1)
        d1 = self.dec1(d1)

        return self.output(d1).squeeze(1)

## Check U-Net output dimensions

The model must preserve the full 12,000-sample temporal resolution required for sample-level P-wave localization.

In [35]:
model = UNet1D().to(device)

waveforms, targets = next(iter(train_loader))

waveforms = waveforms.to(device)
targets = targets.to(device)

with torch.no_grad():
    output = model(waveforms)

print("Input:", waveforms.shape)
print("Target:", targets.shape)
print("Output:", output.shape)

Input: torch.Size([8, 3, 12000])
Target: torch.Size([8, 12000])
Output: torch.Size([8, 12000])


## Define loss and optimizer

Weighted binary cross-entropy and Adam are retained from the baseline CNN to isolate the effect of the new U-Net architecture and broader target.

In [36]:
criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss: Weighted BCEWithLogitsLoss")
print("Optimizer: Adam")
print("Learning rate: 0.001")

Loss: Weighted BCEWithLogitsLoss
Optimizer: Adam
Learning rate: 0.001


## Train the U-Net

The U-Net is trained for 15 epochs using weighted binary cross-entropy and Adam optimization. The model state with the lowest validation loss is retained for subsequent P-wave picking evaluation.

In [37]:
num_epochs = 15

best_val_loss = float("inf")
best_model_state = None

for epoch in range(num_epochs):

    # Training
    model.train()
    train_loss = 0.0

    for waveforms, targets in train_loader:

        waveforms = waveforms.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        outputs = model(waveforms)

        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Validation
    model.eval()
    val_loss = 0.0

    with torch.no_grad():

        for waveforms, targets in val_loader:

            waveforms = waveforms.to(device)
            targets = targets.to(device)

            outputs = model(waveforms)

            loss = criterion(outputs, targets)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    # Save best model
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_model_state = {
            key: value.cpu().clone()
            for key, value in model.state_dict().items()
        }

    print(
        f"Epoch {epoch + 1:02d}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

Epoch 01/15 | Train Loss: 3.3065 | Val Loss: 1.4907
Epoch 02/15 | Train Loss: 1.0590 | Val Loss: 0.8280
Epoch 03/15 | Train Loss: 0.7591 | Val Loss: 0.6293
Epoch 04/15 | Train Loss: 0.6303 | Val Loss: 0.5592
Epoch 05/15 | Train Loss: 0.5692 | Val Loss: 0.4954
Epoch 06/15 | Train Loss: 0.5083 | Val Loss: 0.4716
Epoch 07/15 | Train Loss: 0.5246 | Val Loss: 0.5310
Epoch 08/15 | Train Loss: 0.4813 | Val Loss: 0.4383
Epoch 09/15 | Train Loss: 0.4463 | Val Loss: 0.4309
Epoch 10/15 | Train Loss: 0.4402 | Val Loss: 0.4394
Epoch 11/15 | Train Loss: 0.4127 | Val Loss: 0.4536
Epoch 12/15 | Train Loss: 0.5096 | Val Loss: 0.4293
Epoch 13/15 | Train Loss: 0.4229 | Val Loss: 0.4123
Epoch 14/15 | Train Loss: 0.3970 | Val Loss: 0.3960
Epoch 15/15 | Train Loss: 0.3802 | Val Loss: 0.3943


## Restore the best U-Net checkpoint

The U-Net state with the lowest validation loss is restored before P-wave arrival-time evaluation.

In [38]:
model.load_state_dict(best_model_state)
model.to(device)
model.eval()

print(f"Best validation loss: {best_val_loss:.4f}")

Best validation loss: 0.3943


## Generate validation predictions

The best U-Net checkpoint is used to generate a sample-level P-wave score for every validation waveform.

In [39]:
validation_outputs = []

with torch.no_grad():
    for waveforms, _ in val_loader:
        waveforms = waveforms.to(device)

        logits = model(waveforms)

        validation_outputs.extend(
            logits.cpu().numpy()
        )

validation_outputs = np.asarray(validation_outputs)

validation_true_samples = (
    val_metadata["trace_P_arrival_sample"]
    .values
    .astype(int)
)

print("Validation outputs shape:", validation_outputs.shape)
print("Validation true samples:", len(validation_true_samples))

Validation outputs shape: (1593, 12000)
Validation true samples: 1593


## Extract initial U-Net P-wave picks

The sample with the maximum U-Net output is selected as the initial P-wave arrival prediction. This uses the same decoding method as the baseline CNN.

In [40]:
unet_predicted_samples = np.argmax(
    validation_outputs,
    axis=1
)

unet_errors = (
    np.abs(
        unet_predicted_samples
        - validation_true_samples
    ) / 100.0
)

print(f"Mean absolute error:   {unet_errors.mean():.4f} s")
print(f"Median absolute error: {np.median(unet_errors):.4f} s")
print(f"Within ±0.1 s:         {(unet_errors <= 0.1).mean()*100:.2f}%")
print(f"Within ±0.5 s:         {(unet_errors <= 0.5).mean()*100:.2f}%")
print(f"Within ±1.0 s:         {(unet_errors <= 1.0).mean()*100:.2f}%")
print(f"Within ±2.0 s:         {(unet_errors <= 2.0).mean()*100:.2f}%")

Mean absolute error:   4.1076 s
Median absolute error: 0.1100 s
Within ±0.1 s:         47.58%
Within ±0.5 s:         76.59%
Within ±1.0 s:         81.67%
Within ±2.0 s:         84.18%


## Analyze U-Net predicted arrival locations

The distribution of U-Net-predicted arrival times is examined to identify whether large errors are still caused by late false peaks.

In [41]:
unet_predicted_times = unet_predicted_samples / 100.0

print(
    f"Predicted arrival time: "
    f"mean={unet_predicted_times.mean():.2f}s, "
    f"median={np.median(unet_predicted_times):.2f}s"
)

print("\nPrediction time percentiles:")
for q in [0.50, 0.75, 0.90, 0.95, 0.99]:
    print(
        f"{q*100:.0f}th percentile: "
        f"{np.quantile(unet_predicted_times, q):.2f}s"
    )

print(
    "\nPredictions after 60 s:",
    (unet_predicted_times > 60).sum()
)

print(
    "Predictions after 100 s:",
    (unet_predicted_times > 100).sum()
)

Predicted arrival time: mean=24.02s, median=19.38s

Prediction time percentiles:
50th percentile: 19.38s
75th percentile: 21.94s
90th percentile: 36.21s
95th percentile: 57.89s
99th percentile: 101.64s

Predictions after 60 s: 77
Predictions after 100 s: 18


## Evaluate threshold-based U-Net P-wave picking

The earliest local peak above several absolute CNN-output thresholds is evaluated on the validation set. This uses the same decoding procedure previously tested for the baseline CNN.

In [42]:
from scipy.signal import find_peaks

for threshold in [2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]:

    predicted_samples = []

    for output in validation_outputs:

        peaks, properties = find_peaks(
            output,
            height=threshold,
            distance=50
        )

        if len(peaks) > 0:
            predicted_sample = peaks[0]
        else:
            predicted_sample = np.argmax(output)

        predicted_samples.append(predicted_sample)

    predicted_samples = np.asarray(predicted_samples)

    errors = (
        np.abs(
            predicted_samples - validation_true_samples
        ) / 100.0
    )

    print(
        f"Threshold {threshold:.2f} | "
        f"MAE={errors.mean():.3f}s | "
        f"Median={np.median(errors):.3f}s | "
        f"±0.5s={(errors <= 0.5).mean()*100:.2f}% | "
        f"±1.0s={(errors <= 1.0).mean()*100:.2f}% | "
        f"±2.0s={(errors <= 2.0).mean()*100:.2f}%"
    )

Threshold 2.00 | MAE=5.123s | Median=0.850s | ±0.5s=18.20% | ±1.0s=57.44% | ±2.0s=67.23%
Threshold 2.50 | MAE=4.576s | Median=0.650s | ±0.5s=24.36% | ±1.0s=65.85% | ±2.0s=71.81%
Threshold 3.00 | MAE=3.998s | Median=0.590s | ±0.5s=30.70% | ±1.0s=71.69% | ±2.0s=75.71%
Threshold 3.50 | MAE=3.550s | Median=0.540s | ±0.5s=41.37% | ±1.0s=76.77% | ±2.0s=79.41%
Threshold 4.00 | MAE=3.258s | Median=0.360s | ±0.5s=63.03% | ±1.0s=79.85% | ±2.0s=82.30%
Threshold 4.50 | MAE=3.175s | Median=0.160s | ±0.5s=73.07% | ±1.0s=81.48% | ±2.0s=83.80%
Threshold 5.00 | MAE=3.359s | Median=0.130s | ±0.5s=76.15% | ±1.0s=82.55% | ±2.0s=85.00%


## Evaluate the final U-Net P-wave picker on the test set

The U-Net trained with the 0.30-second Gaussian target is evaluated using the validation-selected absolute output threshold of 5.0. The held-out test set is used only for the final performance assessment.

In [43]:
# Generate test-set U-Net outputs

test_outputs = []

with torch.no_grad():
    for waveforms, _ in test_loader:
        waveforms = waveforms.to(device)

        logits = model(waveforms)

        test_outputs.extend(
            logits.cpu().numpy()
        )

test_outputs = np.asarray(test_outputs)

test_true_samples = (
    test_metadata["trace_P_arrival_sample"]
    .values
    .astype(int)
)

print("Test outputs shape:", test_outputs.shape)

Test outputs shape: (1646, 12000)


## Extract final U-Net P-wave picks

The earliest local peak above the validation-selected threshold of 5.0 is used as the final U-Net P-wave prediction.

In [44]:
final_threshold = 5.0

test_predicted_samples_unet = []

for output in test_outputs:

    peaks, properties = find_peaks(
        output,
        height=final_threshold,
        distance=50
    )

    if len(peaks) > 0:
        predicted_sample = peaks[0]
    else:
        predicted_sample = np.argmax(output)

    test_predicted_samples_unet.append(
        predicted_sample
    )

test_predicted_samples_unet = np.asarray(
    test_predicted_samples_unet
)

test_errors_unet = (
    np.abs(
        test_predicted_samples_unet
        - test_true_samples
    ) / 100.0
)

print(f"Mean absolute error:   {test_errors_unet.mean():.4f} s")
print(f"Median absolute error: {np.median(test_errors_unet):.4f} s")
print(f"Within ±0.1 s:         {(test_errors_unet <= 0.1).mean()*100:.2f}%")
print(f"Within ±0.5 s:         {(test_errors_unet <= 0.5).mean()*100:.2f}%")
print(f"Within ±1.0 s:         {(test_errors_unet <= 1.0).mean()*100:.2f}%")
print(f"Within ±2.0 s:         {(test_errors_unet <= 2.0).mean()*100:.2f}%")

Mean absolute error:   3.8543 s
Median absolute error: 0.1000 s
Within ±0.1 s:         50.91%
Within ±0.5 s:         78.49%
Within ±1.0 s:         83.29%
Within ±2.0 s:         85.18%


## Save final U-Net P-wave picking results

The final held-out test predictions and timing errors are saved for comparison with the baseline CNN and STA/LTA picker.

In [ ]:
from pathlib import Path

results_dir = Path("../results/metrics")
results_dir.mkdir(parents=True, exist_ok=True)

stage2_unet_results = pd.DataFrame({
    "trace_name": test_metadata["trace_name"].values,
    "source_id": test_metadata["source_id"].values,
    "annotated_p_sample": test_true_samples,
    "predicted_p_sample": test_predicted_samples_unet,
    "annotated_p_time_s": test_true_samples / 100.0,
    "predicted_p_time_s": test_predicted_samples_unet / 100.0,
    "abs_error_s": test_errors_unet
})

output_path = results_dir / "stage2_unet_picking.csv"

stage2_unet_results.to_csv(
    output_path,
    index=False
)

print(f"Saved: {output_path}")
print(f"Rows: {len(stage2_unet_results)}")

Saved: results/metrics/stage2_unet_picking.csv
Rows: 1646


## Final Stage 2 model comparison

The final held-out test performance of STA/LTA, the baseline 1D CNN, and the U-Net P-wave picker is compared using arrival-time error metrics.

In [46]:
stage2_final_comparison = pd.DataFrame({
    "Model": [
        "STA/LTA",
        "1D CNN",
        "1D U-Net"
    ],
    "MAE_s": [
        2.7246,
        4.9452,
        3.8543
    ],
    "Median_error_s": [
        0.2500,
        0.1100,
        0.1000
    ],
    "Within_0.1s_pct": [
        30.78,
        49.15,
        50.91
    ],
    "Within_0.5s_pct": [
        63.42,
        82.02,
        78.49
    ],
    "Within_1.0s_pct": [
        72.15,
        83.96,
        83.29
    ],
    "Within_2.0s_pct": [
        76.42,
        84.99,
        85.18
    ]
})

stage2_final_comparison

,Model,MAE_s,Median_error_s,Within_0.1s_pct,Within_0.5s_pct,Within_1.0s_pct,Within_2.0s_pct
0,STA/LTA,2.7246,0.25,30.78,63.42,72.15,76.42
1,1D CNN,4.9452,0.11,49.15,82.02,83.96,84.99
2,1D U-Net,3.8543,0.10,50.91,78.49,83.29,85.18


## Stage 2 results interpretation

STA/LTA achieves the lowest mean absolute error but has lower tolerance-based P-wave picking accuracy.

The baseline 1D CNN provides strong timing accuracy, with a median error of 0.11 seconds and 82.02% of picks within ±0.5 seconds.

The 1D U-Net reduces the CNN's mean absolute error from 4.95 seconds to 3.85 seconds and achieves the lowest median error of 0.10 seconds. It also gives the highest proportion of picks within ±2 seconds at 85.18%.

The U-Net therefore provides an improvement over the baseline CNN, particularly in reducing large timing errors, although STA/LTA remains competitive in mean absolute error.

## Save final Stage 2 model comparison

The final held-out test performance of STA/LTA, the baseline 1D CNN, and the 1D U-Net is saved as a compact summary table.

In [ ]:
from pathlib import Path

results_dir = Path("../results/metrics")
results_dir.mkdir(parents=True, exist_ok=True)

stage2_results = pd.DataFrame({
    "Model": [
        "STA/LTA",
        "1D CNN",
        "1D U-Net"
    ],
    "MAE_s": [
        2.7246,
        4.9452,
        3.8543
    ],
    "Median_error_s": [
        0.2500,
        0.1100,
        0.1000
    ],
    "Within_0.1s_pct": [
        30.78,
        49.15,
        50.91
    ],
    "Within_0.5s_pct": [
        63.42,
        82.02,
        78.49
    ],
    "Within_1.0s_pct": [
        72.15,
        83.96,
        83.29
    ],
    "Within_2.0s_pct": [
        76.42,
        84.99,
        85.18
    ]
})

output_path = results_dir / "stage2_results.csv"

stage2_results.to_csv(
    output_path,
    index=False
)

print(f"Saved: {output_path}")
print(stage2_results)

Saved: results/metrics/stage2_results.csv
      Model   MAE_s  Median_error_s  Within_0.1s_pct  Within_0.5s_pct  \
0   STA/LTA  2.7246            0.25            30.78            63.42   
1    1D CNN  4.9452            0.11            49.15            82.02   
2  1D U-Net  3.8543            0.10            50.91            78.49   

   Within_1.0s_pct  Within_2.0s_pct  
0            72.15            76.42  
1            83.96            84.99  
2            83.29            85.18  


## Save final project results summary

The final held-out test results from Stage 1 earthquake detection and Stage 2 P-wave picking are consolidated into a single CSV file.

In [ ]:
from pathlib import Path
import pandas as pd

results_dir = Path("../results/metrics")
results_dir.mkdir(parents=True, exist_ok=True)

final_results = pd.DataFrame([
    # Stage 1
    {
        "Stage": "Stage 1",
        "Task": "Earthquake Detection",
        "Model": "STA/LTA",
        "Accuracy": 0.6977,
        "Precision": 0.9859,
        "Recall": 0.6798,
        "F1": 0.8047,
        "MAE_s": None,
        "Median_error_s": None,
        "Within_0.5s_pct": None,
        "Within_1.0s_pct": None,
        "Within_2.0s_pct": None
    },
    {
        "Stage": "Stage 1",
        "Task": "Earthquake Detection",
        "Model": "Random Forest",
        "Accuracy": 0.9599,
        "Precision": 0.9752,
        "Recall": 0.9812,
        "F1": 0.9782,
        "MAE_s": None,
        "Median_error_s": None,
        "Within_0.5s_pct": None,
        "Within_1.0s_pct": None,
        "Within_2.0s_pct": None
    },
    {
        "Stage": "Stage 1",
        "Task": "Earthquake Detection",
        "Model": "1D CNN",
        "Accuracy": 0.9304,
        "Precision": 0.9388,
        "Recall": 0.9885,
        "F1": 0.9630,
        "MAE_s": None,
        "Median_error_s": None,
        "Within_0.5s_pct": None,
        "Within_1.0s_pct": None,
        "Within_2.0s_pct": None
    },

    # Stage 2
    {
        "Stage": "Stage 2",
        "Task": "P-wave Picking",
        "Model": "STA/LTA",
        "Accuracy": None,
        "Precision": None,
        "Recall": None,
        "F1": None,
        "MAE_s": 2.7246,
        "Median_error_s": 0.2500,
        "Within_0.5s_pct": 63.42,
        "Within_1.0s_pct": 72.15,
        "Within_2.0s_pct": 76.42
    },
    {
        "Stage": "Stage 2",
        "Task": "P-wave Picking",
        "Model": "1D CNN",
        "Accuracy": None,
        "Precision": None,
        "Recall": None,
        "F1": None,
        "MAE_s": 4.9452,
        "Median_error_s": 0.1100,
        "Within_0.5s_pct": 82.02,
        "Within_1.0s_pct": 83.96,
        "Within_2.0s_pct": 84.99
    },
    {
        "Stage": "Stage 2",
        "Task": "P-wave Picking",
        "Model": "1D U-Net",
        "Accuracy": None,
        "Precision": None,
        "Recall": None,
        "F1": None,
        "MAE_s": 3.8543,
        "Median_error_s": 0.1000,
        "Within_0.5s_pct": 78.49,
        "Within_1.0s_pct": 83.29,
        "Within_2.0s_pct": 85.18
    }
])

output_path = results_dir / "final_project_results.csv"

final_results.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(final_results)

Saved: results/metrics/final_project_results.csv
     Stage                  Task          Model  Accuracy  Precision  Recall  \
0  Stage 1  Earthquake Detection        STA/LTA    0.6977     0.9859  0.6798   
1  Stage 1  Earthquake Detection  Random Forest    0.9599     0.9752  0.9812   
2  Stage 1  Earthquake Detection         1D CNN    0.9304     0.9388  0.9885   
3  Stage 2        P-wave Picking        STA/LTA       NaN        NaN     NaN   
4  Stage 2        P-wave Picking         1D CNN       NaN        NaN     NaN   
5  Stage 2        P-wave Picking       1D U-Net       NaN        NaN     NaN   

       F1   MAE_s  Median_error_s  Within_0.5s_pct  Within_1.0s_pct  \
0  0.8047     NaN             NaN              NaN              NaN   
1  0.9782     NaN             NaN              NaN              NaN   
2  0.9630     NaN             NaN              NaN              NaN   
3     NaN  2.7246            0.25            63.42            72.15   
4     NaN  4.9452            0.11  